# 06 Model Training and Comparison

This notebook trains and compares the four models promised in the proposal:

- Logistic Regression
- Random Forest
- XGBoost
- Isolation Forest

It uses the chronological splits created in `05_chronological_split.ipynb`.

Primary target: `warning_12h`

Important rule: the test set is scored only. Model selection and threshold selection must use the validation set. Final threshold selection and event-level evaluation are handled in Step 7.

In [1]:
# 1. Import libraries and confirm environment

import sys
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = exc

print("Python executable:", sys.executable)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("XGBoost available:", XGBOOST_AVAILABLE)
if not XGBOOST_AVAILABLE:
    print("XGBoost import error:", XGBOOST_IMPORT_ERROR)

Python executable: c:\Users\user\Desktop\metropt_predictive_maintenance_starter\.venv\Scripts\python.exe
pandas: 2.2.2
numpy: 1.26.4
XGBoost available: True


## 1. Configuration

If your laptop is slow, reduce `MAX_TRAIN_ROWS_FOR_FITTING`.  
Set it to `None` only if you want to fit models on the full training set.

Validation and test evaluation still uses the full chronological validation/test sets.

In [1]:
# 2. Configuration

RANDOM_STATE = 42
PRIMARY_TARGET = "warning_12h"

# Laptop-safe default. Set to None for full training data fitting.
MAX_TRAIN_ROWS_FOR_FITTING = 250_000

RF_N_ESTIMATORS = 150
XGB_N_ESTIMATORS = 300

print("Primary target:", PRIMARY_TARGET)
print("Maximum training rows for fitting:", MAX_TRAIN_ROWS_FOR_FITTING)

Primary target: warning_12h
Maximum training rows for fitting: 250000


## 2. Set project paths

In [2]:
# 3. Define project paths

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DATA_DIR = PROCESSED_DATA_DIR / "splits"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_PREDICTIONS_DIR = PROJECT_ROOT / "outputs" / "predictions"
MODELS_DIR = PROJECT_ROOT / "models"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Split data folder:", SPLIT_DATA_DIR)
print("Models folder:", MODELS_DIR)
print("Predictions folder:", OUTPUT_PREDICTIONS_DIR)

NameError: name 'Path' is not defined

## 3. Load chronological split datasets

In [ ]:
# 4. Load X/y datasets

required_files = {
    "X_train": SPLIT_DATA_DIR / "X_train_12h.csv",
    "y_train": SPLIT_DATA_DIR / "y_train_12h.csv",
    "X_val": SPLIT_DATA_DIR / "X_validation_12h.csv",
    "y_val": SPLIT_DATA_DIR / "y_validation_12h.csv",
    "X_test": SPLIT_DATA_DIR / "X_test_12h.csv",
    "y_test": SPLIT_DATA_DIR / "y_test_12h.csv",
    "validation_full": SPLIT_DATA_DIR / "validation_12h.csv",
    "test_full": SPLIT_DATA_DIR / "test_12h.csv"
}

missing = [name for name, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing split files: "
        + ", ".join(missing)
        + ". Run 05_chronological_split.ipynb first."
    )

X_train = pd.read_csv(required_files["X_train"])
y_train = pd.read_csv(required_files["y_train"])[PRIMARY_TARGET].astype(int)

X_val = pd.read_csv(required_files["X_val"])
y_val = pd.read_csv(required_files["y_val"])[PRIMARY_TARGET].astype(int)

X_test = pd.read_csv(required_files["X_test"])
y_test = pd.read_csv(required_files["y_test"])[PRIMARY_TARGET].astype(int)

validation_full = pd.read_csv(required_files["validation_full"])
test_full = pd.read_csv(required_files["test_full"])

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

: 

: 

: 

: 

: 

## 4. Prepare feature names and numeric matrices

In [ ]:
# 5. Sanitize feature names and ensure numeric data


def sanitize_feature_name(name):
    safe = str(name)
    for ch in ["[", "]", "<", ">", "{", "}", ":", ","]:
        safe = safe.replace(ch, "_")
    safe = safe.replace(" ", "_")
    safe = safe.replace("/", "_")
    safe = safe.replace("\\", "_")
    return safe


def make_unique_names(names):
    seen = {}
    unique_names = []
    for name in names:
        base = sanitize_feature_name(name)
        if base not in seen:
            seen[base] = 0
            unique_names.append(base)
        else:
            seen[base] += 1
            unique_names.append(f"{base}_{seen[base]}")
    return unique_names


original_feature_names = list(X_train.columns)
safe_feature_names = make_unique_names(original_feature_names)

feature_name_mapping = pd.DataFrame({
    "original_feature_name": original_feature_names,
    "safe_feature_name": safe_feature_names
})

feature_name_mapping.to_csv(OUTPUT_TABLES_DIR / "model_feature_name_mapping.csv", index=False)

X_train.columns = safe_feature_names
X_val.columns = safe_feature_names
X_test.columns = safe_feature_names

for dataset in [X_train, X_val, X_test]:
    for col in dataset.columns:
        dataset[col] = pd.to_numeric(dataset[col], errors="coerce")
    dataset.replace([np.inf, -np.inf], np.nan, inplace=True)

# Use float32 to reduce memory use.
X_train = X_train.astype("float32")
X_val = X_val.astype("float32")
X_test = X_test.astype("float32")

with open(MODELS_DIR / "model_feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(safe_feature_names, f, indent=2)

print("Number of features:", len(safe_feature_names))
feature_name_mapping.head()

: 

: 

: 

: 

: 

## 5. Create laptop-safe fitting sample

This keeps all warning rows and samples normal rows if the training set is too large.

In [ ]:
# 6. Create fitting sample


def create_fitting_sample(X, y, max_rows=None, random_state=42):
    if max_rows is None or len(X) <= max_rows:
        sample_summary = {
            "training_rows_original": len(X),
            "training_rows_used_for_fitting": len(X),
            "sampling_used": False,
            "positive_rows_used": int(y.sum()),
            "normal_rows_used": int((y == 0).sum())
        }
        return X.copy(), y.copy(), sample_summary

    positive_idx = y[y == 1].index
    normal_idx = y[y == 0].index

    positive_count = len(positive_idx)
    normal_sample_size = max_rows - positive_count

    if normal_sample_size <= 0:
        sampled_idx = positive_idx
    else:
        sampled_normal_idx = pd.Series(normal_idx).sample(
            n=min(normal_sample_size, len(normal_idx)),
            random_state=random_state
        ).values
        sampled_idx = np.concatenate([positive_idx.values, sampled_normal_idx])

    sampled_idx = pd.Index(sampled_idx).sort_values()

    X_sample = X.loc[sampled_idx].copy()
    y_sample = y.loc[sampled_idx].copy()

    sample_summary = {
        "training_rows_original": len(X),
        "training_rows_used_for_fitting": len(X_sample),
        "sampling_used": True,
        "positive_rows_used": int(y_sample.sum()),
        "normal_rows_used": int((y_sample == 0).sum())
    }

    return X_sample, y_sample, sample_summary


X_fit, y_fit, fitting_sample_summary = create_fitting_sample(
    X_train,
    y_train,
    max_rows=MAX_TRAIN_ROWS_FOR_FITTING,
    random_state=RANDOM_STATE
)

fitting_sample_summary_df = pd.DataFrame([fitting_sample_summary])
fitting_sample_summary_df["positive_percentage_used"] = (
    fitting_sample_summary_df["positive_rows_used"] /
    fitting_sample_summary_df["training_rows_used_for_fitting"] * 100
)

fitting_sample_summary_df.to_csv(OUTPUT_TABLES_DIR / "model_fitting_sample_summary.csv", index=False)

fitting_sample_summary_df

: 

: 

: 

: 

: 

## 6. Define evaluation helper functions

In [ ]:
# 7. Evaluation helper functions


def safe_roc_auc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan


def safe_pr_auc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return average_precision_score(y_true, y_score)
    except Exception:
        return np.nan


def evaluate_binary_predictions(y_true, y_score, y_pred, model_name, threshold_strategy):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_score = np.asarray(y_score)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "model": model_name,
        "threshold_strategy": threshold_strategy,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "pr_auc": safe_pr_auc(y_true, y_score),
        "roc_auc": safe_roc_auc(y_true, y_score),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    }


def model_safe_name(model_name):
    return (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("+", "plus")
        .replace("(", "")
        .replace(")", "")
    )

: 

: 

: 

: 

: 

## 7. Train Logistic Regression

In [ ]:
# 8. Train Logistic Regression

logistic_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        C=1.0,
        solver="liblinear",
        random_state=RANDOM_STATE
    ))
])

print("Training Logistic Regression...")
logistic_model.fit(X_fit, y_fit)
joblib.dump(logistic_model, MODELS_DIR / "logistic_regression_model.joblib")
print("Saved:", MODELS_DIR / "logistic_regression_model.joblib")

: 

: 

: 

: 

: 

## 8. Train Random Forest

In [ ]:
# 9. Train Random Forest

random_forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=12,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

print("Training Random Forest...")
random_forest_model.fit(X_fit, y_fit)
joblib.dump(random_forest_model, MODELS_DIR / "random_forest_model.joblib")
print("Saved:", MODELS_DIR / "random_forest_model.joblib")

: 

: 

: 

: 

: 

## 9. Train XGBoost

If XGBoost is unavailable, install it with `%pip install xgboost`, restart the kernel and rerun this notebook.

In [ ]:
# 10. Train XGBoost

trained_xgboost = False

if XGBOOST_AVAILABLE:
    positive_count = int(y_fit.sum())
    normal_count = int((y_fit == 0).sum())
    scale_pos_weight = normal_count / positive_count if positive_count > 0 else 1.0

    xgboost_model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=XGB_N_ESTIMATORS,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            tree_method="hist",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])

    print("Training XGBoost...")
    print("scale_pos_weight:", round(scale_pos_weight, 3))
    xgboost_model.fit(X_fit, y_fit)
    joblib.dump(xgboost_model, MODELS_DIR / "xgboost_model.joblib")
    trained_xgboost = True
    print("Saved:", MODELS_DIR / "xgboost_model.joblib")
else:
    print("XGBoost not available. Skipping XGBoost model.")

: 

: 

: 

: 

: 

## 10. Train Isolation Forest

Isolation Forest is used as an unsupervised anomaly-detection comparator and is fitted mainly on normal training records.

In [ ]:
# 11. Train Isolation Forest

normal_train_mask = y_fit == 0
X_fit_normal = X_fit.loc[normal_train_mask].copy()

MAX_ISOLATION_NORMAL_ROWS = 200_000

if len(X_fit_normal) > MAX_ISOLATION_NORMAL_ROWS:
    X_fit_normal = X_fit_normal.sample(
        n=MAX_ISOLATION_NORMAL_ROWS,
        random_state=RANDOM_STATE
    )

isolation_imputer = SimpleImputer(strategy="median")
X_fit_normal_imputed = isolation_imputer.fit_transform(X_fit_normal)

train_warning_rate = y_fit.mean()
contamination = max(0.001, min(0.10, train_warning_rate if train_warning_rate > 0 else 0.01))

isolation_model = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination=contamination,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training Isolation Forest on normal records...")
print("Normal rows used:", X_fit_normal.shape[0])
print("Contamination:", round(contamination, 5))

isolation_model.fit(X_fit_normal_imputed)

isolation_bundle = {
    "imputer": isolation_imputer,
    "model": isolation_model,
    "feature_names": safe_feature_names,
    "contamination": contamination
}

joblib.dump(isolation_bundle, MODELS_DIR / "isolation_forest_model.joblib")
print("Saved:", MODELS_DIR / "isolation_forest_model.joblib")

: 

: 

: 

: 

: 

## 11. Generate validation and test predictions

These prediction files are needed for Step 7 threshold selection and event-level evaluation.

In [ ]:
# 12. Prediction helper functions

trained_models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}

if trained_xgboost:
    trained_models["XGBoost"] = xgboost_model


def supervised_scores(model, X):
    return model.predict_proba(X)[:, 1]


def isolation_scores_and_predictions(bundle, X):
    X_imp = bundle["imputer"].transform(X)
    model = bundle["model"]

    raw_anomaly_score = -model.decision_function(X_imp)

    min_score = np.nanmin(raw_anomaly_score)
    max_score = np.nanmax(raw_anomaly_score)

    if max_score > min_score:
        score = (raw_anomaly_score - min_score) / (max_score - min_score)
    else:
        score = np.zeros_like(raw_anomaly_score)

    pred = (model.predict(X_imp) == -1).astype(int)
    return score, pred


validation_predictions = pd.DataFrame({"y_true": y_val.values})
test_predictions = pd.DataFrame({"y_true": y_test.values})

for metadata_col in ["timestamp", "Timestamp", "time", "Time", "datetime", "Datetime"]:
    if metadata_col in validation_full.columns:
        validation_predictions.insert(0, "timestamp", validation_full[metadata_col].values)
        test_predictions.insert(0, "timestamp", test_full[metadata_col].values)
        break

for metadata_col in ["warning_12h_event_id", "failure_event_id"]:
    if metadata_col in validation_full.columns:
        validation_predictions[metadata_col] = validation_full[metadata_col].values
    if metadata_col in test_full.columns:
        test_predictions[metadata_col] = test_full[metadata_col].values

validation_metrics = []
test_default_metrics = []

for model_name, model in trained_models.items():
    safe_name = model_safe_name(model_name)

    val_score = supervised_scores(model, X_val)
    test_score = supervised_scores(model, X_test)

    val_pred_default = (val_score >= 0.50).astype(int)
    test_pred_default = (test_score >= 0.50).astype(int)

    validation_predictions[f"{safe_name}_score"] = val_score
    validation_predictions[f"{safe_name}_pred_default_0_50"] = val_pred_default

    test_predictions[f"{safe_name}_score"] = test_score
    test_predictions[f"{safe_name}_pred_default_0_50"] = test_pred_default

    validation_metrics.append(
        evaluate_binary_predictions(
            y_val, val_score, val_pred_default, model_name, "default probability threshold 0.50"
        )
    )

    test_default_metrics.append(
        evaluate_binary_predictions(
            y_test, test_score, test_pred_default, model_name, "default probability threshold 0.50"
        )
    )

iso_val_score, iso_val_pred = isolation_scores_and_predictions(isolation_bundle, X_val)
iso_test_score, iso_test_pred = isolation_scores_and_predictions(isolation_bundle, X_test)

validation_predictions["isolation_forest_score"] = iso_val_score
validation_predictions["isolation_forest_pred_default"] = iso_val_pred

test_predictions["isolation_forest_score"] = iso_test_score
test_predictions["isolation_forest_pred_default"] = iso_test_pred

validation_metrics.append(
    evaluate_binary_predictions(
        y_val, iso_val_score, iso_val_pred, "Isolation Forest", "model contamination threshold"
    )
)

test_default_metrics.append(
    evaluate_binary_predictions(
        y_test, iso_test_score, iso_test_pred, "Isolation Forest", "model contamination threshold"
    )
)

validation_predictions_path = OUTPUT_PREDICTIONS_DIR / "validation_predictions_all_models.csv"
test_predictions_path = OUTPUT_PREDICTIONS_DIR / "test_predictions_all_models.csv"

validation_predictions.to_csv(validation_predictions_path, index=False)
test_predictions.to_csv(test_predictions_path, index=False)

print("Saved validation predictions:", validation_predictions_path)
print("Saved test predictions:", test_predictions_path)

: 

: 

: 

: 

: 

## 12. Save validation model comparison table

In [ ]:
# 13. Save model comparison metrics

validation_model_comparison = pd.DataFrame(validation_metrics).sort_values(
    by=["f2", "pr_auc"],
    ascending=False
)

test_default_model_comparison = pd.DataFrame(test_default_metrics).sort_values(
    by=["f2", "pr_auc"],
    ascending=False
)

validation_model_comparison.to_csv(
    OUTPUT_TABLES_DIR / "validation_model_comparison_record_level.csv",
    index=False
)

test_default_model_comparison.to_csv(
    OUTPUT_TABLES_DIR / "test_default_model_comparison_record_level.csv",
    index=False
)

validation_model_comparison

: 

: 

: 

: 

: 

## 13. Plot validation Precision-Recall curves

In [ ]:
# 14. Plot validation precision-recall curves

plt.figure(figsize=(9, 6))

score_columns = [col for col in validation_predictions.columns if col.endswith("_score")]

for score_col in score_columns:
    model_label = score_col.replace("_score", "").replace("_", " ").title()
    precision, recall, _ = precision_recall_curve(
        validation_predictions["y_true"],
        validation_predictions[score_col]
    )
    ap = average_precision_score(
        validation_predictions["y_true"],
        validation_predictions[score_col]
    )
    plt.plot(recall, precision, label=f"{model_label} AP={ap:.3f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation Precision-Recall Curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / "validation_precision_recall_curves.png", dpi=300)
plt.show()

: 

: 

: 

: 

: 

## 14. Plot validation confusion matrices

In [ ]:
# 15. Plot confusion matrices for validation predictions

prediction_columns = [
    col for col in validation_predictions.columns
    if col.startswith(("logistic", "random", "xgboost")) and col.endswith("_pred_default_0_50")
]
prediction_columns += ["isolation_forest_pred_default"]

for pred_col in prediction_columns:
    if pred_col not in validation_predictions.columns:
        continue

    cm = confusion_matrix(
        validation_predictions["y_true"],
        validation_predictions[pred_col],
        labels=[0, 1]
    )

    display_label = pred_col.replace("_pred_default_0_50", "").replace("_pred_default", "").replace("_", " ").title()

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Normal", "Warning"]
    )

    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, values_format="d")
    plt.title(f"Validation confusion matrix: {display_label}")
    plt.tight_layout()
    filename = f"validation_confusion_matrix_{pred_col}.png"
    plt.savefig(OUTPUT_FIGURES_DIR / filename, dpi=300)
    plt.show()

: 

: 

: 

: 

: 

## 15. Plot validation model comparison

In [ ]:
# 16. Plot validation model comparison by F2 score

plot_metrics = validation_model_comparison.copy()

plt.figure(figsize=(9, 5))
plt.bar(plot_metrics["model"], plot_metrics["f2"])
plt.ylabel("Validation F2 score")
plt.title("Validation model comparison using F2 score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / "validation_model_comparison_f2.png", dpi=300)
plt.show()

: 

: 

: 

: 

: 

## 16. Save feature importance outputs

In [ ]:
# 17. Feature importance helper


def save_top_importance(importance_values, feature_names, model_name, output_prefix, top_n=30):
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importance_values
    })

    importance_df["absolute_importance"] = importance_df["importance"].abs()
    importance_df = importance_df.sort_values("absolute_importance", ascending=False)

    table_path = OUTPUT_TABLES_DIR / f"{output_prefix}_top_feature_importance.csv"
    importance_df.to_csv(table_path, index=False)

    top_df = importance_df.head(top_n).sort_values("absolute_importance", ascending=True)

    plt.figure(figsize=(10, 8))
    plt.barh(top_df["feature"], top_df["importance"])
    plt.xlabel("Importance")
    plt.title(f"Top {top_n} features: {model_name}")
    plt.tight_layout()
    fig_path = OUTPUT_FIGURES_DIR / f"{output_prefix}_top_feature_importance.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

    print("Saved:", table_path)
    print("Saved:", fig_path)

    return importance_df


lr_coefficients = logistic_model.named_steps["model"].coef_[0]
logistic_importance = save_top_importance(
    lr_coefficients,
    safe_feature_names,
    "Logistic Regression coefficients",
    "logistic_regression",
    top_n=25
)

rf_importances = random_forest_model.named_steps["model"].feature_importances_
rf_importance = save_top_importance(
    rf_importances,
    safe_feature_names,
    "Random Forest feature importance",
    "random_forest",
    top_n=25
)

if trained_xgboost:
    xgb_importances = xgboost_model.named_steps["model"].feature_importances_
    xgb_importance = save_top_importance(
        xgb_importances,
        safe_feature_names,
        "XGBoost feature importance",
        "xgboost",
        top_n=25
    )

: 

: 

: 

: 

: 

## 17. Save model training summary

In [ ]:
# 18. Save model training summary

trained_model_names = list(trained_models.keys()) + ["Isolation Forest"]

model_training_summary = pd.DataFrame({
    "item": [
        "Primary target",
        "Models trained",
        "Training rows available",
        "Training rows used for fitting",
        "Validation rows evaluated",
        "Test rows scored",
        "Number of feature columns",
        "Random state",
        "Class imbalance handling",
        "Test set use in this notebook",
        "Next step"
    ],
    "value": [
        PRIMARY_TARGET,
        ", ".join(trained_model_names),
        len(X_train),
        len(X_fit),
        len(X_val),
        len(X_test),
        len(safe_feature_names),
        RANDOM_STATE,
        "class weighting / scale_pos_weight / Isolation Forest contamination",
        "Only scored with default thresholds; not used for model or threshold selection",
        "Step 7 threshold selection and event-level evaluation"
    ]
})

model_training_summary.to_csv(
    OUTPUT_TABLES_DIR / "model_training_summary.csv",
    index=False
)

model_training_summary

: 

: 

: 

: 

: 

## 18. Save model registry

In [ ]:
# 19. Save model registry

registry_records = [
    {
        "model": "Logistic Regression",
        "model_file": str(MODELS_DIR / "logistic_regression_model.joblib"),
        "type": "supervised classification"
    },
    {
        "model": "Random Forest",
        "model_file": str(MODELS_DIR / "random_forest_model.joblib"),
        "type": "supervised classification"
    },
    {
        "model": "Isolation Forest",
        "model_file": str(MODELS_DIR / "isolation_forest_model.joblib"),
        "type": "unsupervised anomaly detection"
    }
]

if trained_xgboost:
    registry_records.append({
        "model": "XGBoost",
        "model_file": str(MODELS_DIR / "xgboost_model.joblib"),
        "type": "supervised classification"
    })

model_registry = pd.DataFrame(registry_records)
model_registry["validation_predictions_file"] = str(validation_predictions_path)
model_registry["test_predictions_file"] = str(test_predictions_path)

model_registry.to_csv(OUTPUT_TABLES_DIR / "model_registry.csv", index=False)

model_registry

: 

: 

: 

: 

: 

## 19. Completion checklist

In [ ]:
# 20. Completion checklist

completion_checklist = pd.DataFrame({
    "task": [
        "Chronological split files loaded",
        "Feature names sanitised",
        "Training fitting sample created",
        "Logistic Regression trained",
        "Random Forest trained",
        "XGBoost trained or gracefully skipped",
        "Isolation Forest trained",
        "Validation predictions saved",
        "Test predictions saved",
        "Validation model comparison saved",
        "Precision-recall curve saved",
        "Confusion matrices saved",
        "Feature importance tables saved",
        "Model registry saved"
    ],
    "status": [
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete" if trained_xgboost else "Skipped - XGBoost unavailable",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete"
    ]
})

completion_checklist.to_csv(
    OUTPUT_TABLES_DIR / "model_training_completion_checklist.csv",
    index=False
)

completion_checklist

: 

: 

: 

: 

: 

## 20. Confirm saved outputs

In [ ]:
# 21. Confirm saved outputs

print("Saved model-related tables:")
for file in sorted(OUTPUT_TABLES_DIR.glob("*.csv")):
    if "model" in file.name or "importance" in file.name or "feature_name_mapping" in file.name:
        print("-", file.name)

print("\nSaved model-related figures:")
for file in sorted(OUTPUT_FIGURES_DIR.glob("*.png")):
    if "model" in file.name or "confusion" in file.name or "precision_recall" in file.name or "importance" in file.name:
        print("-", file.name)

print("\nSaved prediction files:")
for file in sorted(OUTPUT_PREDICTIONS_DIR.glob("*.csv")):
    print("-", file.name)

print("\nSaved model files:")
for file in sorted(MODELS_DIR.glob("*.joblib")):
    print("-", file.name)

: 

: 

: 

: 

: 